In [2]:
from pathlib import Path
import html
import shutil
import tempfile
import time

from IPython.display import HTML, display

from olympdev.frontend.html.job import render_job
from olympdev.isolate import IsolateFactory
from olympdev.problem.types import ProblemConfiguration, ProblemTest
from olympdev.resources import asset_file
from olympdev.storage.filestorage import FileStorage
from olympdev.worker.cache import MemoryCacheStorage
from olympdev.worker.executor import JobExecutor
from olympdev.worker.jobs.simple_run import SimpleRunJobResult
from olympdev.worker.pipeline import NextJobStatus, PipelineContext, run_pipeline
from olympdev.worker.pipelines.problem_testing import ProblemTestingPipeline
from olympdev.worker.verdict import Verdict


PROBLEM = Path("example-contest/aplusb").resolve()
GXX = shutil.which("g++")
assert GXX is not None, "g++ is required"

_STATIC_SAMPLES = ("tests/sample-01.txt", "tests/sample-02.txt")
_STATIC_TESTS = ("tests/manual-01.txt",)
_GENERATED_TESTS = (
    ("gen -c=1e3", "random-01.txt"),
    ("gen -c=1e6", "random-02.txt"),
    ("gen -c=1e9", "random-03.txt"),
)
SOLUTIONS = (
    ("solutions/sol-ac.cpp", "main"),
    ("solutions/sol-wa.cpp", "wa"),
    ("solutions/sol-tl.cpp", "tl"),
)

storage = FileStorage(Path(tempfile.mkdtemp(prefix="aplusb-")) / "storage")
isolate_factory = IsolateFactory(1)


def aplusb_configuration() -> ProblemConfiguration:
    return ProblemConfiguration(
        time_limit=1000,
        memory_limit=512,
        input_file_name="input.txt",
        output_file_name="output.txt",
        samples=[ProblemTest(input_file=path) for path in _STATIC_SAMPLES],
        tests=[
            ProblemTest(input_file=_STATIC_TESTS[0]),
            *[ProblemTest(gen_cmd=cmd, name=name) for cmd, name in _GENERATED_TESTS],
        ],
    )


def make_testing_digest(source: Path) -> str:
    with asset_file("testlib.h") as testlib:
        return ProblemTestingPipeline.make_input_digest(
            storage,
            sources=[(source, "main.cpp")],
            command=[GXX, "-O2", "-o", "solution", 0],
            checker_sources=[
                (PROBLEM / "source" / "ncmp.cpp", "ncmp.cpp"),
                (testlib, "testlib.h"),
            ],
            checker_command=[GXX, "-O2", "-o", "checker", 0],
            model_sources=[(PROBLEM / "solutions" / "sol-ac.cpp", "model.cpp")],
            model_command=[GXX, "-O2", "-o", "model", 0],
            generator_sources=[
                (PROBLEM / "source" / "gen.cpp", "gen.cpp"),
                (testlib, "testlib.h"),
            ],
            generator_command=[GXX, "-O2", "-o", "gen", 0],
            validator_sources=[
                (PROBLEM / "source" / "valid.cpp", "valid.cpp"),
                (testlib, "testlib.h"),
            ],
            validator_command=[GXX, "-O2", "-o", "valid", 0],
            configuration=aplusb_configuration(),
            problem_dir=PROBLEM,
            problem_name=PROBLEM.name,
        )


def time_solutions(*, executor=None, pipeline_prefix: str = "aplusb"):
    """Run every solution in SOLUTIONS. Returns (runs, total_wall_s)."""
    runs = []
    total_started = time.perf_counter()
    for relpath, kind in SOLUTIONS:
        source = PROBLEM / relpath
        pipeline = ProblemTestingPipeline(f"{pipeline_prefix}-{source.stem}")
        context = PipelineContext(
            input_digest=make_testing_digest(source),
            storage=storage,
            isolate_factory=isolate_factory,
        )
        started = time.perf_counter()
        run_pipeline(pipeline, context, executor=executor)
        wall_s = time.perf_counter() - started
        runs.append({
            "name": source.name,
            "kind": kind,
            "wall_s": wall_s,
            "result": pipeline.get_result(storage),
        })
    return runs, time.perf_counter() - total_started


def run_pipeline_verbose(pipeline, context: PipelineContext) -> None:
    """Sequential runner: render each issued job as HTML after execute."""
    while True:
        nxt = pipeline.get_next_job(context)
        if nxt is NextJobStatus.FINISHED:
            status = pipeline.get_status().value
            reason = pipeline.get_failure_reason()
            digest = pipeline.get_output_digest() or ""
            reason_html = f"<div style='margin-top:4px'>reason: {html.escape(str(reason))}</div>" if reason else ""
            digest_html = (
                f"<div style='margin-top:4px'>result <code>{html.escape(digest)}</code></div>"
                if digest
                else ""
            )
            display(HTML(
                "<div style='font:13px/1.45 ui-sans-serif,system-ui,sans-serif;"
                "padding:10px 12px;margin:8px 0 14px;border-radius:10px;"
                "border:1px solid color-mix(in srgb, CanvasText 14%, transparent)'>"
                f"FINISHED · pipeline <b>{html.escape(status)}</b>"
                f"{reason_html}{digest_html}"
                "</div>"
            ))
            return
        if nxt is NextJobStatus.WAITING:
            raise RuntimeError(
                f"pipeline {pipeline.pipeline_id!r} is waiting with no in-flight jobs"
            )
        nxt.job.execute(context.job_context(nxt.input_digest))
        display(HTML(nxt.job.describe(render_job)))


def _fmt_time(seconds: float) -> str:
    ms = seconds * 1000
    if ms < 10:
        return f"{ms:.1f} ms"
    return f"{ms:.0f} ms"


def _fmt_memory(kb: int) -> str:
    if kb >= 1024:
        return f"{kb / 1024:.1f} MB"
    return f"{kb} KB"


def _run_usage(run_digest: str | None) -> tuple[str, str]:
    if run_digest is None:
        return "—", "—"
    result = SimpleRunJobResult.from_storage(storage, run_digest)
    if result is None:
        return "—", "—"
    memory_kb = result.meta.cg_memory_kb or result.meta.max_rss_kb
    # Isolate `time` is CPU. Sleepy solutions hit the wall-time limit with
    # almost no CPU, so TL would look like a few milliseconds.
    elapsed = (
        result.meta.time_wall
        if result.meta.status == "TO"
        else result.meta.time
    )
    return _fmt_time(elapsed), _fmt_memory(memory_kb)


_VERDICT_COLORS = {
    Verdict.OK: "#166534",
    Verdict.WA: "#991b1b",
    Verdict.TL: "#9a3412",
    Verdict.ML: "#9a3412",
    Verdict.RE: "#7c2d12",
    Verdict.CE: "#6b21a8",
    Verdict.CF: "#1e3a8a",
    Verdict.FL: "#334155",
    Verdict.PT: "#1d4ed8",
}


def render_solutions_table(runs: list[dict]) -> str:
    """runs: dicts with name, kind, wall_s, result (ProblemTestingResult | None)."""
    test_names: list[str] = []
    for run in runs:
        result = run["result"]
        if result is None:
            continue
        names = [test.name for test in result.tests]
        if not test_names:
            test_names = names
        elif names != test_names:
            raise ValueError("solutions produced different test lists")

    header_cells = ["<th class='od-rt-test'>Test</th>"]
    for run in runs:
        result = run["result"]
        overall = "—" if result is None else result.verdict.value
        color = _VERDICT_COLORS.get(None if result is None else result.verdict, "#334155")
        header_cells.append(
            "<th>"
            f"<div class='od-rt-sol'>{html.escape(run['name'])}</div>"
            f"<div class='od-rt-kind'>{html.escape(run['kind'])}</div>"
            f"<div class='od-rt-overall' style='color:{color}'>{html.escape(overall)}</div>"
            f"<div class='od-rt-wall'>wall {_fmt_time(run['wall_s'])}</div>"
            "</th>"
        )

    body_rows = []
    for index, test_name in enumerate(test_names):
        cells = [f"<th class='od-rt-test'>{html.escape(test_name)}</th>"]
        for run in runs:
            result = run["result"]
            if result is None or index >= len(result.tests):
                cells.append("<td class='od-rt-empty'>—</td>")
                continue
            test = result.tests[index]
            time_s, memory = _run_usage(test.run_digest)
            color = _VERDICT_COLORS.get(test.verdict, "#334155")
            cells.append(
                "<td>"
                f"<div class='od-rt-verdict' style='color:{color}'>{html.escape(test.verdict.value)}</div>"
                f"<div class='od-rt-usage'>{html.escape(time_s)}</div>"
                f"<div class='od-rt-usage'>{html.escape(memory)}</div>"
                "</td>"
            )
        body_rows.append("<tr>" + "".join(cells) + "</tr>")

    if not test_names:
        body_rows.append(
            "<tr><td class='od-rt-empty' colspan='{}'>no tests</td></tr>".format(1 + len(runs))
        )

    return f"""
<style>
  .od-rt {{
    font: 13px/1.45 ui-sans-serif, system-ui, sans-serif;
    border-collapse: collapse;
    width: 100%;
    margin: 8px 0 14px;
  }}
  .od-rt th, .od-rt td {{
    border: 1px solid color-mix(in srgb, CanvasText 14%, transparent);
    padding: 8px 10px;
    vertical-align: top;
    text-align: left;
  }}
  .od-rt thead th {{
    background: color-mix(in srgb, CanvasText 5%, Canvas);
    font-weight: 600;
  }}
  .od-rt-test {{
    white-space: nowrap;
    font-family: ui-monospace, SFMono-Regular, Menlo, Consolas, monospace;
    font-size: 12px;
    font-weight: 600;
  }}
  .od-rt tbody th.od-rt-test {{ font-weight: 500; }}
  .od-rt-sol {{ font-weight: 650; }}
  .od-rt-kind, .od-rt-wall, .od-rt-usage {{
    color: color-mix(in srgb, CanvasText 58%, transparent);
    font-size: 11px;
  }}
  .od-rt-kind {{ text-transform: uppercase; letter-spacing: 0.04em; margin-top: 1px; }}
  .od-rt-overall {{ font-weight: 700; margin-top: 4px; }}
  .od-rt-verdict {{ font-weight: 700; }}
  .od-rt-empty {{ color: color-mix(in srgb, CanvasText 45%, transparent); }}
</style>
<table class="od-rt">
  <thead><tr>{''.join(header_cells)}</tr></thead>
  <tbody>{''.join(body_rows)}</tbody>
</table>
"""

ModuleNotFoundError: No module named 'jinja2'

In [3]:
source = PROBLEM / "solutions" / "sol-ac.cpp"
input_digest = make_testing_digest(source)
pipeline = ProblemTestingPipeline("aplusb-main")
context = PipelineContext(
    input_digest=input_digest,
    storage=storage,
    isolate_factory=isolate_factory,
)
run_pipeline_verbose(pipeline, context)

isolate --cg --cleanup failed (exit 2): This box was not initialized properly
isolate --cleanup failed (exit 2): This box was initialized with an incompatible control group mode


isolate init with quota failed, retrying without quota: Cannot set disk quota: quotas have not been enabled for this filesystem
isolate --cg --cleanup failed (exit 2): This box was not initialized properly
isolate --cleanup failed (exit 2): This box was initialized with an incompatible control group mode


Result,4218bbca2e1cb4442e4ef530b1bb1442dd358e84f6a3802ba5ae3454f2306d01


Result,d56d30b9f5e0e3b132985b792e4bed30bc3056008c72b9f3acb6faa91caba3d8


Result,2806fd83f6592187b06bbdc6e3e83c35aa556182874e259d6d3161ab1c2832df


Result,b2cbc21adda41c7dc49b9df74283588271824134595c675751df15e8664aa9af


... (24 outputs left out)

Result,0fd52230f51c5f6c6d27ddaa5aad851375ffb35ab457aa9202a693ed2cf673aa


Result,87deac67fc5a5560c6b03afc94683c0ae87d11de065f6a522d03a94ac3fe84b5


Result,85ff315ea895d8d9b05c9317bb66429ce242ad9cc798d3cf02202aee9516cc37


Result,282e3a6be63f836ef0a8bbb556c55c70d67e3b38045986eeadecd635ed807edf


In [4]:
uncached_runs, uncached_total = time_solutions(pipeline_prefix="nocache")
cached_runs, cached_total = time_solutions(
    executor=JobExecutor(MemoryCacheStorage()),
    pipeline_prefix="cache",
)

display(HTML(render_solutions_table(uncached_runs)))

speedup = uncached_total / cached_total if cached_total else float("inf")
cmp_rows = "".join(
    "<tr>"
    f"<td>{html.escape(uncached['name'])}</td>"
    f"<td>{html.escape(uncached['kind'])}</td>"
    f"<td><b>{html.escape('—' if uncached['result'] is None else uncached['result'].verdict.value)}</b></td>"
    f"<td>{html.escape(_fmt_time(uncached['wall_s']))}</td>"
    f"<td>{html.escape(_fmt_time(cached['wall_s']))}</td>"
    f"<td>{uncached['wall_s'] / cached['wall_s']:.2f}×</td>"
    "</tr>"
    for uncached, cached in zip(uncached_runs, cached_runs)
)
display(HTML(
    "<div style='font:13px/1.45 ui-sans-serif,system-ui,sans-serif;"
    "padding:10px 12px;margin:8px 0;border-radius:10px;"
    "border:1px solid color-mix(in srgb, CanvasText 14%, transparent)'>"
    "<div style='margin-bottom:8px;font-weight:650'>Wall time: no cache vs shared MemoryCacheStorage</div>"
    "<div style='margin-bottom:10px;color:color-mix(in srgb, CanvasText 58%, transparent);font-size:12px'>"
    "Cached batch shares one cache across solutions, so checker / validator / generator / model "
    "jobs hit after the first solution."
    "</div>"
    "<table style='border-collapse:collapse;width:100%'>"
    "<thead><tr>"
    "<th style='text-align:left;padding:4px 8px 4px 0'>solution</th>"
    "<th style='text-align:left;padding:4px 8px 4px 0'>type</th>"
    "<th style='text-align:left;padding:4px 8px 4px 0'>verdict</th>"
    "<th style='text-align:left;padding:4px 8px 4px 0'>no cache</th>"
    "<th style='text-align:left;padding:4px 8px 4px 0'>with cache</th>"
    "<th style='text-align:left;padding:4px 0'>speedup</th>"
    "</tr></thead>"
    f"<tbody>{cmp_rows}"
    "<tr style='font-weight:650'>"
    "<td>total</td><td></td><td></td>"
    f"<td>{html.escape(_fmt_time(uncached_total))}</td>"
    f"<td>{html.escape(_fmt_time(cached_total))}</td>"
    f"<td>{speedup:.2f}×</td>"
    "</tr>"
    "</tbody></table></div>"
))

Test,sol-ac.cppmainOKwall 10589 ms,sol-wa.cppwaWAwall 10688 ms,sol-tl.cpptlTLwall 20742 ms
tests/sample-01.txt,OK1.0 ms580 KB,WA2.0 ms580 KB,OK2.0 ms856 KB
tests/sample-02.txt,OK1.0 ms580 KB,WA1.0 ms580 KB,OK2.0 ms712 KB
tests/manual-01.txt,OK1.0 ms580 KB,WA1.0 ms580 KB,TL2100 ms580 KB
random-01.txt,OK1.0 ms592 KB,WA1.0 ms580 KB,TL2100 ms580 KB
random-02.txt,OK1.0 ms588 KB,WA1.0 ms580 KB,TL2100 ms768 KB
random-03.txt,OK1.0 ms592 KB,WA1.0 ms580 KB,TL2100 ms580 KB


solution,type,verdict,no cache,with cache,speedup
sol-ac.cpp,main,OK,10589 ms,11526 ms,0.92×
sol-wa.cpp,wa,WA,10688 ms,450 ms,23.74×
sol-tl.cpp,tl,TL,20742 ms,9031 ms,2.30×
total,,,42027 ms,21013 ms,2.00×
